In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import json

print("All imports successful")
print("CUDA available:", torch.cuda.is_available())

All imports successful
CUDA available: True


In [4]:
import os
from datasets import Dataset
import transformers
transformers.logging.set_verbosity_error()

In [6]:
import os
print(os.getcwd())

/teamspace/studios/this_studio


In [7]:
with open("train.json") as f:
    train_data = json.load(f)

with open("valid.json") as f:
    val_data = json.load(f)

print(f"Train: {len(train_data)} examples")
print(f"Val: {len(val_data)} examples")
print("\nSample example:")
print(json.dumps(train_data[0], indent=2))

Train: 33190 examples
Val: 4710 examples

Sample example:
{
  "example_id": 1,
  "conv_id": "hit:0_conv:1",
  "utterance_idx": 2,
  "input_to_model": {
    "emotion": "sad",
    "text": "I remember going to see the fireworks with my best friend . It was the first time we ever spent time alone together . Although there was a lot of people , we felt like the only people in the world ."
  },
  "target": "Was this a friend you were in love with , or just a best friend ?"
}


In [8]:
def format_prompt(example):
    emotion = example['input_to_model']['emotion']
    text = example['input_to_model']['text']
    target = example['target']
    
    prompt = f"""<s>[INST] You are an empathetic conversational assistant. The user is feeling {emotion}.
User: {text}
Respond with empathy. [/INST] {target} </s>"""
    
    return {"text": prompt}

train_formatted = [format_prompt(ex) for ex in train_data]
val_formatted = [format_prompt(ex) for ex in val_data]

print("Sample formatted prompt:")
print(train_formatted[0]['text'])

Sample formatted prompt:
<s>[INST] You are an empathetic conversational assistant. The user is feeling sad.
User: I remember going to see the fireworks with my best friend . It was the first time we ever spent time alone together . Although there was a lot of people , we felt like the only people in the world .
Respond with empathy. [/INST] Was this a friend you were in love with , or just a best friend ? </s>


In [9]:
import os
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")  # Set HF_TOKEN in your environment before running

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.environ["HF_TOKEN"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)

print("Model loaded successfully")
print(f"Model device: {next(model.parameters()).device}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully
Model device: cuda:0


In [10]:
#setting up LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 27,262,976 || all params: 7,268,995,072 || trainable%: 0.3751


In [11]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

sft_config = SFTConfig(
    output_dir="./mistral-empathetic-v4",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    bf16=True,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=50,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    dataset_text_field="text",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/33190 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/33190 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/4710 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4710 [00:00<?, ? examples/s]

{'loss': '3.756', 'grad_norm': '1.812', 'learning_rate': '4.9e-05', 'entropy': '1.89', 'num_tokens': '7.4e+04', 'mean_token_accuracy': '0.4651', 'epoch': '0.0241'}
{'loss': '1.563', 'grad_norm': '1.508', 'learning_rate': '9.9e-05', 'entropy': '1.506', 'num_tokens': '1.463e+05', 'mean_token_accuracy': '0.6568', 'epoch': '0.0482'}
{'loss': '1.342', 'grad_norm': '1.125', 'learning_rate': '9.985e-05', 'entropy': '1.352', 'num_tokens': '2.191e+05', 'mean_token_accuracy': '0.6677', 'epoch': '0.07231'}
{'loss': '1.342', 'grad_norm': '0.8945', 'learning_rate': '9.938e-05', 'entropy': '1.344', 'num_tokens': '2.938e+05', 'mean_token_accuracy': '0.6651', 'epoch': '0.09641'}
{'loss': '1.318', 'grad_norm': '0.9648', 'learning_rate': '9.86e-05', 'entropy': '1.324', 'num_tokens': '3.687e+05', 'mean_token_accuracy': '0.6714', 'epoch': '0.1205'}
{'loss': '1.298', 'grad_norm': '0.957', 'learning_rate': '9.752e-05', 'entropy': '1.308', 'num_tokens': '4.419e+05', 'mean_token_accuracy': '0.6758', 'epoch': 

TrainOutput(global_step=2075, training_loss=1.3218207347823914, metrics={'train_runtime': 2859.104, 'train_samples_per_second': 11.609, 'train_steps_per_second': 0.726, 'total_flos': 2.089993297540055e+17, 'train_loss': 1.3218207347823914})

In [12]:
model.save_pretrained("./mistral-empathetic-v4")
tokenizer.save_pretrained("./mistral-empathetic-v4")
print("Model saved!")

Model saved!


In [ ]:
with open("test.json") as f:
    test_data = json.load(f)

results = []
OUTPUT_FILE = "mistral_responses_v4.json"

for i, ex in enumerate(test_data):
    emotion = ex['input_to_model']['emotion']
    text    = ex['input_to_model']['text']
    target  = ex['target']

    prompt = (
        f"<s>[INST] You are an empathetic conversational assistant. "
        f"The user is feeling {emotion}.\n"
        f"User: {text}\n"
        f"Respond with empathy. [/INST]"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    results.append({
        "example_id": ex['example_id'],
        "emotion": emotion,
        "input": text,
        "target": target,
        "generated": generated
    })

    if len(results) % 100 == 0:
        print(f"Progress: {len(results)}/{len(test_data)}")
        with open(OUTPUT_FILE, "w") as f:
            json.dump(results, f, indent=2)

with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"Done! Generated {len(results)} responses")

Progress: 100/4329
Progress: 200/4329
Progress: 300/4329
Progress: 400/4329
Progress: 500/4329
Progress: 600/4329
Progress: 700/4329
Progress: 800/4329
Progress: 900/4329
Progress: 1000/4329
Progress: 1100/4329
Progress: 1200/4329
Progress: 1300/4329
Progress: 1400/4329
Progress: 1500/4329
Progress: 1600/4329
Progress: 1700/4329
Progress: 1800/4329
Progress: 1900/4329
Progress: 2000/4329
Progress: 2100/4329
Progress: 2200/4329
Progress: 2300/4329
Progress: 2400/4329
Progress: 2500/4329
Progress: 2600/4329
Progress: 2700/4329
Progress: 2800/4329
Progress: 2900/4329
Progress: 3000/4329
Progress: 3100/4329
Progress: 3200/4329
Progress: 3300/4329
Progress: 3400/4329
Progress: 3500/4329
Progress: 3600/4329


In [1]:
import os
import json
import torch
import transformers
transformers.logging.set_verbosity_error()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("Imports done")
print("CUDA:", torch.cuda.is_available())

Imports done
CUDA: True


In [2]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")  # Set HF_TOKEN in your environment before running

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained("./mistral-empathetic-v4")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)
model = PeftModel.from_pretrained(model, "./mistral-empathetic-v4")
print("Model loaded!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded!


In [3]:
with open("test.json") as f:
    test_data = json.load(f)

with open("mistral_responses_v4.json") as f:
    results = json.load(f)

print(f"Resuming from example {len(results)}")

for i, ex in enumerate(test_data):
    if i < len(results):
        continue

    emotion = ex['input_to_model']['emotion']
    text    = ex['input_to_model']['text']
    target  = ex['target']

    prompt = (
        f"<s>[INST] You are an empathetic conversational assistant. "
        f"The user is feeling {emotion}.\n"
        f"User: {text}\n"
        f"Respond with empathy. [/INST]"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    results.append({
        "example_id": ex['example_id'],
        "emotion": emotion,
        "input": text,
        "target": target,
        "generated": generated
    })

    if len(results) % 100 == 0:
        print(f"Progress: {len(results)}/{len(test_data)}")
        with open("mistral_responses_v4.json", "w") as f:
            json.dump(results, f, indent=2)

with open("mistral_responses_v4.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Done! Generated {len(results)} responses")

Resuming from example 3600
Progress: 3700/4329
Progress: 3800/4329
Progress: 3900/4329
Progress: 4000/4329
Progress: 4100/4329
Progress: 4200/4329
Progress: 4300/4329
Done! Generated 4329 responses


In [4]:
for r in results[10:15]:
    print(f"Emotion: {r['emotion']}")
    print(f"Input: {r['input']}")
    print(f"Target: {r['target']}")
    print(f"Generated: {r['generated']}")
    print("-" * 80)

Emotion: happy
Input: so my friend ' s baby fell down and scraped his knee yesterday and i was trying so hard to cheer him up
Target: That is nice of you . What did you do ?
Generated: Oh no ! I hope he is okay .
--------------------------------------------------------------------------------
Emotion: happy
Input: so my friend ' s baby fell down and scraped his knee yesterday and i was trying so hard to cheer him up <SOC> That is nice of you . What did you do ? <SOC> i was singing nursery songs : )
Target: Which one ?
Generated: That is so sweet . I bet he felt better .
--------------------------------------------------------------------------------
Emotion: happy
Input: my husband lost a job but i ' m hoping he can find a full time job soon
Target: He will , I have faith .
Generated: I ' m sorry to hear that . I hope he finds a new job soon .
--------------------------------------------------------------------------------
Emotion: happy
Input: my husband lost a job but i ' m hoping he

In [5]:
print(f"Total responses: {len(results)}")

Total responses: 4329
